In [ ]:
import time
for i in range(10):
    print('test')
    time.sleep(2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
#sys.path.append('/d/ret1/Taylor/jupyter_notebooks/Research/slug2')  # Path to slug2 directory
sys.path.append('/project/galaxies/tjuchau/software/Slug/slug2')

import os
#home_directory = "/d/ret1/Taylor/jupyter_notebooks/Research" 

from Functions import *

import glob
import re
#slug_path = "/d/ret1/Taylor/jupyter_notebooks/Research/slug2/bin/slug"
slug_path = "/project/galaxies/tjuchau/software/Slug/slug2/bin/slug"
import slugpy
#wd = '/d/ret1/Taylor/jupyter_notebooks/Research'
wd = '/project/galaxies/tjuchau'
slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'
os.chdir('/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files')

In [ ]:
dt = 1e5
N = 10
masses = [200, 2000,5000,10000,20000,50000]
def try_float(x):
    '''
    Try to convert an item or an array of items to float.
    If conversion fails, returns original value(s).
    
    Parameters
    ----------
    x : scalar or array-like
        Item(s) to convert.
    
    Returns
    -------
    float, array of floats, or original input if conversion fails.
    '''
    # If x is array-like, attempt vectorized conversion
    if isinstance(x, (list, np.ndarray)):
        try:
            return np.array(x, dtype=float)
        except ValueError:
            return x
    else:
        # Scalar input
        try:
            return float(x)
        except ValueError:
            return x
            
def do_alex_model(model_name, mass,dt = 1e6, N=1000, tracks = 'mist_2016_vvcrit_00', spec_synth = 'sb99'):
    '''Replication of Alex's model'''
    input_file = f"{model_name}.slugin"
    
    with open(input_file, 'w') as f:
        f.write(f'model_name {model_name}\n')
        f.write(f'out_dir {slug_output_dir}\n') #TJ this is where the output files will be written to
        f.write(f'verbosity 1\n') #TJ level of printed outputs while running (0=only warnings/errors) (1=some outputs) (2=lots of outputs)
        ##################################################################
        # Parameters controlling simulation execution and physical model #
        ##################################################################
        f.write(f'sim_type cluster\n') #TJ must be either galaxy or cluster (defaults to galaxy)
        f.write(f'n_trials {N}\n') #TJ total number of model clusters to run
        #f.write(f'checkpoint_interval = 100\n') #TJ create checkpoint after this many trials (default to no checkpointing)
        f.write(f'time_step {dt}\n') #TJ simulation runs for 1million years before computing new values
        #f.write(f'start_time 1.0e6\n') #TJ default start time is the same as timestep
        f.write(f'end_time 1.0e7\n') #TJ how long does each simulation run for in years
        #f.write(f'sfr 0.001\n') #TJ star formation rate, ignored for sim types = cluster
        #f.write(f'sfh sfh.txt\n') #TJ star formation history, ignored for sim types = cluster
        f.write(f'cluster_mass {mass}\n') #TJ cluster mass in solar masses, ignored for sim type = galaxy
        #f.write(f'redshift 0\n') #TJ defaults to 0
        ##################################################################
        # Parameters controlling simulation outputs #
        ##################################################################
        f.write(f'out_cluster 1\n') #TJ output cluster properties? default = 1
        f.write(f'out_cluster_phot 1\n') #TJ output cluster photometry? (must specify filters also)
        f.write(f'out_cluster_spec 1\n') #TJ output cluster spectroscopy? *adds significant computation time*
        f.write(f'out_cluster_yield 1\n') #TJ output cluster nucleosynthesis yields?
        #f.write(f'out_integrated 1\n') #TJ output integrated properties of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_phot 1\n') #TJ output integrated photometry of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_spec 1\n') #TJ output integrated spectroscopy of galaxy? ignored for sim types = cluster
        #f.write(f'out_integrated_yield 1\n') #TJ output integrated chemical yields of galaxy? ignored for sim types = cluster
        f.write(f'output_mode ascii\n') #TJ can be either binary, ascii, or fits
        #####################################################################
        # Parameters controlling the physical models used for stars         #
        #####################################################################
        #f.write(f'imf lib/imf/chabrier.imf\n') #TJ what imf to use? defaults to chabrier 2001
        #f.write(f'cmf lib/cmf/slug_default.cmf\n') #TJ cluster mass function for galaxies, ignored for sim types = cluster
        f.write(f'clf lib/clf/nodisrupt.clf\n') #TJ cluster lifetime Default: lib/clf/slug_default.clf (dN/dt ~ t^-1.9)
        f.write(f'tracks {tracks}\n') #TJ choose the stellar track. Defaults to geneva_2013_vvcrit_00
        f.write(f'atmospheres lib/atmospheres\n') #TJ directory of the stellar atmospheres
        f.write(f'specsyn_mode {spec_synth}\n') #TJ Spectral synthesis mode, describing which models to use for stellar atmospheres allowed values below
        # -- planck (treat stars as blackbodies)
        # -- kurucz (use Kurucz atmospheres, as compiled by Lejeune+ 1997)
        # -- kurucz+hillier (use Hillier models for WR stars, kurucz for all others)
        # -- kurucz+pauldrach (use Pauldrach models for OB stars, kurucz for others)
        # -- sb99 (emulate starburst99 -- Pauldrach for OB stars, Hillier for WR stars, kurucz for others) This is the default value
        f.write(f'clust_frac 1.0\n') #TJ fraction of stars born in clusters (always 1.0 for sim types = cluster)
        f.write(f'min_stoch_mass 0.08\n') #TJ minimum stochastically sampled mass. Everything below is considered to be continuously sampled
        #f.write(f'metallicity       1.0\n') #TJ metalicity function. If tracks is specified, this should be omitted
        #####################################################################
        # Parameters controlling extinction                                 #
        #####################################################################
        f.write(f'A_V lib/avdist/slug_default.av\n') #TJ set extinction function
        f.write(f'extinction_curve lib/extinct/MW_EXT_SLUG.dat\n') #TJ shape of extinction curve
        f.write(f'nebular_extinction_factor lib/avdist/neb_factor_default.av\n') #TJ use a different extinction law for nebulae
        #####################################################################
        # Parameters controlling nebular emission                           #
        #####################################################################
        f.write(f'compute_nebular 1\n') #TJ compute nebular emission specifically?
        #f.write(f'atomic_data lib/atomic\n') #TJ atomic information, defaults to lib/atomic
        #f.write(f'nebular_no_metals 0\n') #TJ 1 would be to turn off nebular metal emission (includes He), 0 means leave metals on
        #f.write(f'nebular_den 1.0e2\n') #TJ hydrogen density (default is 100)
        #f.write(f'nebular_temp -1.0\n') #TJ nebular temperature, default is -1, if negative, temp will be calculated from cloudy
        f.write(f'nebular_logU -2.5\n') #TJ logU representing ionization parameter
        f.write(f'nebular_phi 0.73\n') #TJ fraction of ionizing photons that are absorbed by Hydrogen atoms
        #############################################
        # Parameters describing photometric filters #
        #############################################
        f.write(f'phot_bands JWST_F150W, JWST_F187N, JWST_F300M, QH0\n') #TJ list of filters for photometric results
        #f.write(f'filters lib/filters\n') #TJ directory for filter information to be read from (defaults to lib/filters)
        #f.write(f'phot_mode Lnu\n') #TJ what units should the photometry results print in? (defaults to Lnu)
        ############################################
        # Parameters controlling yield calculation #
        ############################################
        f.write(f'yield_dir lib/yields\n') #TJ directory for yield files
        
        # are available:
        # 
        
        # 
        f.write(f'yield_mode sukhbold16+karakas16+doherty14\n') #TJ Model to use for yield calculation. Currently the following models accepted:
        # -- sukhbold16 = Solar metallicity type II SN yields from Sukhbold et al. (2016, ApJ, 821, 38); no other yields
        # -- # karakas16+doherty14 = metallicity-dependent AGB star yields from Karakas & Lugaro (2016, ApJ, 825, 26), and super- 
        #                                                                                 AGB star yields from Doherty+ (2014, MNRAS, 437, 195)
        # -- sukhbold16+karakas16+doherty14 = sukhbold16 used for SNII, karakas16+doherty14 for AGB
        f.write(f'\n')
        wd = os.getcwd().split('medbow')[-1]

    return wd+f'/{input_file}'


def read_all_files(model_name, slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'):
    '''
    Reads all SLUG output .txt files for a given model_name into a dictionary of numpy arrays.
    Keys are column names, values are data arrays.
    '''
    output = {}
    files = glob.glob(f'{slug_output_dir}/{model_name}_cluster*.txt')
    
    for file in files:
        print(f'reading {file}')
        if file[-5]=='d':
            print(f'skipping {file}')
            continue
        elif file[-9:] == '_spec.txt':
            continue
        with open(file, 'r') as f:
            lines = f.readlines()
        
        # Get column names
        col_names = lines[0].strip().split()
        n_cols_names = len(col_names)
        
        # Parse data lines
        data_entries = []
        
        for line in lines[2:]:
            stripped = line.strip()
            if line.startswith('---------'):
                continue
            entries = stripped.split()
            
            # Pad missing entries with 'nan'
            if len(entries) < n_cols_names:
                entries += ['nan'] * (n_cols_names - len(entries))
            data_entries.append(entries)
        
        # Skip empty files gracefully
        if not data_entries:
            continue
        
        # Convert to numpy array of strings first
        data_array = np.array(data_entries, dtype='U20')
        
        # Convert each column individually to float if possible, else keep as string
        for i, name in enumerate(col_names):
            col = data_array[:, i]
            try:
                col_converted = col.astype(float)
            except ValueError:
                col_converted = col  # keep as string if conversion fails
            
            if name in output:
                print(f"Warning: Duplicate column name '{name}' found. Overwriting previous value.")
            output[name] = col_converted
            
    return output


def compute_paalpha_ew(data):  
    """  
    Compute Paα flux and equivalent width from SLUG output.  

    Args:  
        data (dict): Dictionary with filter fluxes (e.g., 'JWST_F150W').  

    Returns:  
        dict: Paα flux (Jy), continuum (Jy), and EW (Å).  
    """  
    # Extract fluxes (adjust keys if needed)  
    f150 = data['JWST_F150W']  # Continuum filter 1  
    f300 = data['JWST_F300M']  # 300m filter
    f187n = data['JWST_F187N'] # Paα filter  
    f187c = data['JWST_F187N_n'] #continuum around Paa
    f300n = data['JWST_F300M_n'] #continuum around 300
    # Estimate continuum at Paα (1.875 µm) by linear interpolation  
    # Wavelengths (µm) for each filter (central λ from JWST)  
    lambda150 = 1.50  
    lambda187 = 1.875  
    lambda300 = 3.00  

    # Linear fit to continuum (F150W and F200W)  
    slope = (f300 - f150) / (lambda300 - lambda150)  
    continuum = f150 + slope * (lambda187 - lambda150)  
    
    # Subtract continuum to isolate Paα flux  
    paalpha_flux = f187n - continuum  
    paalpha_flux_c = f187n - f187c
    # Compute equivalent width (EW) in Ångströms  
    # EW = Δλ (F_line / F_continuum), where Δλ = F187N filter width (~0.02 µm = 200 Å)  
    f187n_width = 0.02 * 1e4  # Convert µm to Å (1 µm = 1e4 Å)  
    ew = f187n_width * (paalpha_flux / continuum)  
    ew_c = f187n_width * (paalpha_flux_c / f187c)  

    return {'paalpha_flux': paalpha_flux, 'continuum': continuum, 'EW': ew,
            'paalpha_flux_c': paalpha_flux, 'continuum_c': continuum, 'EW_c': ew_c}

def get_EW_using_phot(Fnu_feature, Fnu_cont):
    Fnu_cont *= u.erg/u.s/u.Hz  
    Fnu_feature *= u.erg/u.s/u.Hz  
    feature_filter = "F187N"
    continuum_filters = ['F150W', 'F300M']

    pivot_feat = jwst_pivots[feature_filter]
    pivot_cont = [jwst_pivots[f] for f in continuum_filters]
    
    #TJ convert continuum levels into F_lambda using pivot wavelengths, still need to multiply by dlamda
    fλ_cont = [(Fnu * c / pivot**2).to(u.W / u.m)
                for Fnu, pivot in zip(Fnu_cont, pivot_cont)]
    
    #TJ get mean wavelengths
    cont_wls = [jwst_means[f] for f in continuum_filters]
    line_wl = jwst_means[feature_filter]

    #TJ interpolate continuum values to the feature wavelength
    feature_continuum = np.interp(
        line_wl.value,
        [w.value for w in cont_wls],
        [f.value for f in fλ_cont]
    ) * u.W / u.m
    
    #TJ print continuum if needed
    #print("F_lamda of photo continuum : ", feature_continuum)
    #TJ get filter transmission curve info
    wl, T = get_filter_data(feature_filter)

    #TJ multiply feature F_lambda by dlambda to complete unit conversion
    norm = np.trapezoid(T, wl) / np.max(T)
    cont_in_filter = feature_continuum * norm
    
    #TJ convert feature filter's F_nu into F_lamda
    fλ_feature = ((Fnu_feature * c / pivot_feat**2).to(u.W / u.m))*norm 

    #TJ feature area is only the area above the continuum
    feature_only = fλ_feature - cont_in_filter

    #TJEquivalent width is this area divided by the continuum level
    EW = (feature_only / feature_continuum).to(u.m)

    return EW


def read_slug_spec(spec_file):
    with open(spec_file, 'r') as f:
        lines = f.readlines()
    
    col_names = lines[0].strip().split()
    n_cols = len(col_names)
    
    data_entries = []
    for line in lines[2:]:
        if line.startswith('---------'):
            continue
        
        entries = line.strip().split()
        if len(entries) < n_cols:
            entries += ['nan'] * (n_cols - len(entries))
        
        data_entries.append(entries)
    
    return np.array(data_entries, dtype=float)


def compute_paalpha_ew(wavelength, flux,
                      line_center=18756.0,
                      line_window=200,     # +/- Angstrom around line
                      cont_window=500):    # continuum window
    
    # --- Define regions ---
    line_mask = (wavelength > line_center - line_window) & \
                (wavelength < line_center + line_window)
    
    cont_mask = (
        ((wavelength > line_center - cont_window) & 
         (wavelength < line_center - line_window)) |
        ((wavelength > line_center + line_window) & 
         (wavelength < line_center + cont_window))
    )
    
    if np.sum(cont_mask) < 5:
        return np.nan
    
    # --- Continuum estimate ---
    cont_level = np.median(flux[cont_mask])
    
    if cont_level <= 0:
        return np.nan
    
    # --- Sort line region ---
    wl_line = wavelength[line_mask]
    fl_line = flux[line_mask]
    
    sort_idx = np.argsort(wl_line)
    wl_line = wl_line[sort_idx]
    fl_line = fl_line[sort_idx]
    
    # --- Compute EW ---
    integrand = (fl_line - cont_level) / cont_level
    ew = np.trapezoid(integrand, wl_line)
    
    return ew


def measure_paalpha_ew_from_files(files):
    
    all_ages = []
    all_ews = []
    all_trials = []
    
    for spec_file in files:
        data = read_slug_spec(spec_file)
        
        trial = data[:,0]     # <-- IMPORTANT
        time = data[:,1]
        wavelength = data[:,2]
        flux = data[:,3] + data[:,4]
        
        # Unique combinations of (trial, time)
        unique_pairs = np.unique(np.column_stack((trial, time)), axis=0)
        
        for tr, t in unique_pairs:
            mask = (trial == tr) & (time == t)
            
            wl = wavelength[mask]
            fl = flux[mask]
            
            if len(wl) < 10:
                continue
            
            ew = compute_paalpha_ew(wl, fl)
            
            if not np.isnan(ew):
                all_ages.append(t)
                all_ews.append(ew)
                all_trials.append(tr)
    
    return np.array(all_ages), np.array(all_ews), np.array(all_trials)


def plot_paalpha_ew_vs_age(files):
    
    ages, ews, trials = measure_paalpha_ew_from_files(files)
    
    plt.figure(figsize=(8,6))
    
    # Option 1: scatter all realizations
    plt.scatter(ages, ews, alpha=0.3)
    
    # Option 2 (better): median + scatter envelope
    bins = np.logspace(np.log10(min(ages)), np.log10(max(ages)), 30)
    
    bin_centers = []
    medians = []
    
    for i in range(len(bins)-1):
        mask = (ages >= bins[i]) & (ages < bins[i+1])
        
        if np.sum(mask) > 5:
            bin_centers.append(np.sqrt(bins[i]*bins[i+1]))
            medians.append(np.median(ews[mask]))
    
    plt.plot(bin_centers, medians, color='black', lw=2, label='Median')
    
    plt.xscale('log')
    #plt.yscale('log')
    
    plt.xlabel('Age (yr)')
    plt.ylabel('Paα Equivalent Width (Å)')
    plt.title('Paα EW vs Cluster Age (SLUG)')
    
    plt.legend()
    plt.tight_layout()
    plt.show()
def check_if_npy_file(model_name, slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'):
    check = os.path.exists(f'{slug_output_dir}/{model_name}.npy')
    return check
#files = glob.glob('/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/no_rotation*spec.txt')
#plot_paalpha_ew_vs_age(files)

In [ ]:
file = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/no_rotation20000_cluster_spec.txt'
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

def plot_slug_spectra(spec_file):
    
    # -------- Read file --------
    with open(spec_file, 'r') as f:
        lines = f.readlines()
    
    col_names = lines[0].strip().split()
    n_cols = len(col_names)
    
    data_entries = []
    for line in lines[2:]:
        if line.startswith('---------'):
            continue
        
        entries = line.strip().split()
        
        if len(entries) < n_cols:
            entries += ['nan'] * (n_cols - len(entries))
        
        data_entries.append(entries)
    
    data = np.array(data_entries, dtype=float)
    
    # -------- Extract columns --------
    trial = data[:,0]
    time = data[:,1]
    wavelength = data[:,2]
    flux_neb = data[:,3] + data[:,4] 
    
    # -------- Unique pairs --------
    pairs = np.unique(np.column_stack((trial, time)), axis=0)
    
    # -------- Colormap (by time only) --------
    unique_times = np.unique(time)
    cmap = plt.cm.rainbow
    norm = plt.Normalize(vmin=unique_times.min(), vmax=unique_times.max())
    
    plt.figure(figsize=(8,6))
    
    for tr, t in pairs:
        
        mask = (trial == tr) & (time == t)
        
        wl = wavelength[mask]
        fl = flux_neb[mask]
        
        # sort by wavelength
        sort_idx = np.argsort(wl)
        wl = wl[sort_idx]
        fl = fl[sort_idx]
        
        # restrict to Pa-alpha region or both filters
        region = (wl > 18650) & (wl < 18780)
        #region = (wl > 15000) & (wl < 30000)
        
        if np.sum(region) < 5:
            continue
        
        plt.plot(wl[region], fl[region],
                 color=cmap(norm(t)),
                 alpha=0.2)   # low alpha → reveals density
    
    # -------- Formatting --------
    plt.xlabel('Wavelength (Angstrom)')
    plt.ylabel('L_lambda (nebular)')
    plt.title('SLUG Spectral Evolution (Paα region)')
    
    plt.yscale('log')
    plt.xscale('log')
    
    plt.tight_layout()
    plt.show()
plot_slug_spectra(file)

In [ ]:
dt = 1e6
N = 100
masses = [200, 2000,5000,10000,20000,50000]
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]

for m in masses:
    model_name = f'testing_max_masses_{m}'
    spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
    slug_input_file = do_alex_model(model_name, m, dt = dt, N=N, tracks = 'mist_2016_vvcrit_00', spec_synth='kurucz')
    
    run_command_on_ARCC('/cluster/medbow/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files',
    model_name, f"conda run -n Modeling {slug_path} {slug_input_file}", 
    cpus = 1 if m < 10001 else 4, memory=8, fail_notification=True, finish_notification=False, time=8, conda = "Modeling")
    #os.system(f"conda run -n Modeling {slug_path} {slug_input_file}")


In [ ]:
masses = [200, 2000,5000,10000,20000,50000]
N= 100
dt = 1e5
for m in masses:
    model_name = f'testing_1e5_{m}'
    if not check_if_npy_file(model_name):
        spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
        temp = read_all_files(model_name)
        rows = [dict(zip(temp.keys(), values)) for values in zip(*temp.values())]
        ews = []
        news = []
        print(f'Reading spec file for {model_name}')
        _, spews, _ = measure_paalpha_ew_from_files([spec_file])

        for row in rows:
            news.append(get_EW_using_phot(row["JWST_F187N_n"], [row['JWST_F150W_n'], row['JWST_F300M_n']]).value*1e10)
            ews.append(get_EW_using_phot(row["JWST_F187N"], [row['JWST_F150W'], row['JWST_F300M']]).value*1e10)
        
        data = np.array([temp['Time'], ews, news, spews])
        np.save(f'{slug_output_dir}/{model_name}.npy', data)
    else:
        print('numpy file already exists, moving on...')

# Determine grid size (e.g. 2 rows x 3 columns for 6 plots)
n_cols = 3
n_rows = int(np.ceil(len(masses) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
fig2, ax2 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax2 = ax2.flatten()
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax3 = ax3.flatten()


for i, m in enumerate(masses):
    model_name = f'testing_1e5_{m}'
    data = np.load(f'{slug_output_dir}/{model_name}.npy')
    
    ax = axes[i]
    ax2_i = ax2[i]
    ax3_i = ax3[i]

    chunk_data = []

    for chunk in range(N):
        start = int(1e7/dt)*chunk
        ax.plot(data[0][start:start+int(1e7/dt)], data[1][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')
        ax2_i.plot(data[0][start:start+int(1e7/dt)], data[2][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')
        ax3_i.plot(data[0][start:start+int(1e7/dt)], data[3][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')

        chunk_data.append(data[1][start:start+int(1e7/dt)])
        
    chunk_data = np.array(chunk_data)  # shape: (100, 10)
    
    # Calculate median across chunks for each timestep
    median_y = np.nanmedian(chunk_data, axis=0)
    
    # Overplot median track
    #ax.plot(data[0][start:start+int(1e7/dt)], median_y, color='red', linewidth=3, label='Median')
    
    ax.set_title(model_name)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('EW')
    ax2_i.set_title(model_name)
    ax2_i.set_xlabel('Age (years)')
    ax2_i.set_ylabel('neb EW')
    ax3_i.set_title(model_name)
    ax3_i.set_xlabel('Age (years)')
    ax3_i.set_ylabel('spec EW')

    ax.legend(fontsize=6)
# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
N = 100
dt = 1e6
for m in masses:
    model_name = f'testing_kurucz_{m}'
    spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
    temp = read_all_files(model_name)
    rows = [dict(zip(temp.keys(), values)) for values in zip(*temp.values())]
    ews = []
    news = []
    max_mass = []
    _, spews, _ = measure_paalpha_ew_from_files([spec_file])

    for row in rows:
        news.append(get_EW_using_phot(row["JWST_F187N_n"], [row['JWST_F150W_n'], row['JWST_F300M_n']]).value*1e10)
        ews.append(get_EW_using_phot(row["JWST_F187N"], [row['JWST_F150W'], row['JWST_F300M']]).value*1e10)
        max_mass.append(row['MaxStarMass'])
    data = np.array([temp['Time'], ews, max_mass, spews])
    np.save(f'{slug_output_dir}/{model_name}.npy', data)



# Determine grid size (e.g. 2 rows x 3 columns for 6 plots)
n_cols = 3
n_rows = int(np.ceil(len(masses) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
fig2, ax2 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax2 = ax2.flatten()
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax3 = ax3.flatten()


for i, m in enumerate(masses):
    model_name = f'testing_kurucz_{m}'
    data = np.load(f'{slug_output_dir}/{model_name}.npy')
    
    ax = axes[i]
    ax2_i = ax2[i]
    ax3_i = ax3[i]

    chunk_data = []


    for chunk in range(N):
        start = int(1e7/dt)*chunk
        ax.plot(data[0][start:start+int(1e7/dt)], data[1][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')
        ax2_i.plot(data[0][start:start+int(1e7/dt)], data[2][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')
        ax3_i.plot(data[0][start:start+int(1e7/dt)], data[3][start:start+int(1e7/dt)], linewidth = 1 if N<11 else 0.1, color = colors[chunk] if N<11 else 'black')

        chunk_data.append(data[1][start:start+int(1e7/dt)])
        
    chunk_data = np.array(chunk_data)  # shape: (100, 10)
    
    # Calculate median across chunks for each timestep
    median_y = np.nanmedian(chunk_data, axis=0)
    
    # Overplot median track
    #ax.plot(data[0][start:start+int(1e7/dt)], median_y, color='red', linewidth=3, label='Median')
    
    ax.set_title(model_name)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('EW')
    ax2_i.set_title(model_name)
    ax2_i.set_xlabel('Age (years)')
    ax2_i.set_ylabel('max_star_mass')
    ax3_i.set_title(model_name)
    ax3_i.set_xlabel('Age (years)')
    ax3_i.set_ylabel('spec EW')

    ax.legend(fontsize=6)
# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
dt = 1e6
N = 10
masses = [200, 2000, 5000, 10000, 20000, 50000]
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]
# Determine grid size (e.g. 2 rows x 3 columns for 6 plots)
n_cols = 3
n_rows = int(np.ceil(len(masses) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
fig2, ax2 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax2 = ax2.flatten()
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax3 = ax3.flatten()
fig4, ax4 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax4 = ax4.flatten()
fig5, ax5 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax5 = ax5.flatten()
fig6, ax6 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax6 = ax6.flatten()
for i, m in enumerate(masses):
    model_name = f'no_rotation{m}'
    data = np.load(f'{slug_output_dir}/{model_name}.npy')
    
    ax = axes[i]
    ax2_i = ax2[i]
    ax3_i = ax3[i]
    ax4_i = ax4[i]
    ax5_i = ax5[i]
    ax6_i = ax6[i]
    chunk_data = []


    for chunk in range(N):
        start = int(1e7/dt)*chunk
        ax.plot(data[0][start:start+int(1e7/dt)], data[1][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax.plot(data[0][start:start+int(1e7/dt)], data[2][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        ax2_i.plot(data[0][start:start+int(1e7/dt)], data[3][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax2_i.plot(data[0][start:start+int(1e7/dt)], data[4][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        ax3_i.plot(data[0][start:start+int(1e7/dt)], data[5][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax3_i.plot(data[0][start:start+int(1e7/dt)], data[6][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        ax4_i.plot(data[0][start:start+int(1e7/dt)], data[7][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax4_i.plot(data[0][start:start+int(1e7/dt)], data[8][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        ax5_i.plot(data[0][start:start+int(1e7/dt)], data[9][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax5_i.plot(data[0][start:start+int(1e7/dt)], data[10][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[11][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[12][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        chunk_data.append(data[1][start:start+int(1e7/dt)])
        
    chunk_data = np.array(chunk_data)  # shape: (100, 10)
    
    # Calculate median across chunks for each timestep
    median_y = np.nanmedian(chunk_data, axis=0)
    
    # Overplot median track
    #ax.plot(data[0][start:start+int(1e7/dt)], median_y, color='red', linewidth=3, label='Median')
    
    ax.set_title(model_name)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('EW in Angstroms')
    ax2_i.set_title(model_name)
    ax2_i.set_xlabel('Age (years)')
    ax2_i.set_ylabel('QH0')
    ax3_i.set_title(model_name)
    ax3_i.set_xlabel('Age (years)')
    ax3_i.set_ylabel('f187')
    ax4_i.set_title(model_name)
    ax4_i.set_xlabel('Age (years)')
    ax4_i.set_ylabel('f150 level')
    ax5_i.set_title(model_name)
    ax5_i.set_xlabel('Age (years)')
    ax5_i.set_ylabel('f300')
    ax6_i.set_title(model_name)
    ax6_i.set_xlabel('Age (years)')
    ax6_i.set_ylabel('max_star_mass')
    ax.legend(fontsize=6)
# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
masses = [200, 2000, 5000, 10000, 20000, 50000]
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]
# Determine grid size (e.g. 2 rows x 3 columns for 6 plots)
n_cols = 3
n_rows = int(np.ceil(len(masses) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
fig2, ax2 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax2 = ax2.flatten()
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax3 = ax3.flatten()
fig4, ax4 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax4 = ax4.flatten()
fig5, ax5 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax5 = ax5.flatten()
fig6, ax6 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax6 = ax6.flatten()
for i, m in enumerate(masses):
    model_name = f'rotation{m}'
    data = np.load(f'{slug_output_dir}/{model_name}.npy')
    
    ax = axes[i]
    ax2_i = ax2[i]
    ax3_i = ax3[i]
    ax4_i = ax4[i]
    ax5_i = ax5[i]
    ax6_i = ax6[i]
    chunk_data = []

    dt = 1e6
    N = 10
    for chunk in range(N):
        start = int(1e7/dt)*chunk
        ax.plot(data[0][start:start+int(1e7/dt)], data[1][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax.plot(data[0][start:start+int(1e7/dt)], data[2][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        ax2_i.plot(data[0][start:start+int(1e7/dt)], data[3][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax2_i.plot(data[0][start:start+int(1e7/dt)], data[4][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        ax3_i.plot(data[0][start:start+int(1e7/dt)], data[5][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax3_i.plot(data[0][start:start+int(1e7/dt)], data[6][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        ax4_i.plot(data[0][start:start+int(1e7/dt)], data[7][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax4_i.plot(data[0][start:start+int(1e7/dt)], data[8][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        ax5_i.plot(data[0][start:start+int(1e7/dt)], data[9][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax5_i.plot(data[0][start:start+int(1e7/dt)], data[10][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[11][start:start+int(1e7/dt)], linewidth = 2, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[12][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        chunk_data.append(data[1][start:start+int(1e7/dt)])
        
    chunk_data = np.array(chunk_data)  # shape: (100, 10)
    
    # Calculate median across chunks for each timestep
    median_y = np.nanmedian(chunk_data, axis=0)
    
    # Overplot median track
    #ax.plot(data[0][start:start+int(1e7/dt)], median_y, color='red', linewidth=3, label='Median')
    
    ax.set_title(model_name)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('EW in Angstroms')
    ax2_i.set_title(model_name)
    ax2_i.set_xlabel('Age (years)')
    ax2_i.set_ylabel('QH0')
    ax3_i.set_title(model_name)
    ax3_i.set_xlabel('Age (years)')
    ax3_i.set_ylabel('f187')
    ax4_i.set_title(model_name)
    ax4_i.set_xlabel('Age (years)')
    ax4_i.set_ylabel('f150 level')
    ax5_i.set_title(model_name)
    ax5_i.set_xlabel('Age (years)')
    ax5_i.set_ylabel('f300')
    ax6_i.set_title(model_name)
    ax6_i.set_xlabel('Age (years)')
    ax6_i.set_ylabel('max_star_mass')
    ax.legend(fontsize=6)
# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
dt = 1e5
N = 10
masses = [200, 2000,5000,10000,20000,50000]
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]

for m in masses:
    model_name = f'10_trials_dt_1e5_m_{m}'
    spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
    slug_input_file = do_alex_model(model_name, m, dt = dt, N=N, tracks = 'mist_2016_vvcrit_00', spec_synth='kurucz')
    
    run_command_on_ARCC('/cluster/medbow/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files',
    model_name, f"conda run -n Modeling {slug_path} {slug_input_file}", 
    cpus = 5, memory=8, fail_notification=True, finish_notification=False, time=8, conda = "Modeling")


In [ ]:
#data = read_all_files(model_name)
for m in masses:
    model_name = f'10_trials_dt_1e5_m_{m}'
    spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
    #plot_slug_spectra(spec_file)
    if not check_if_npy_file(model_name):
        spec_file = f'/project/galaxies/tjuchau/data_files/misc_data/slug_outputs/{model_name}_cluster_spec.txt'
        temp = read_all_files(model_name)
        rows = [dict(zip(temp.keys(), values)) for values in zip(*temp.values())]
        ews = []
        news = []
        print(f'Reading spec file for {model_name}')
        _, spews, _ = measure_paalpha_ew_from_files([spec_file])

        for row in rows:
            news.append(get_EW_using_phot(row["JWST_F187N_n"], [row['JWST_F150W_n'], row['JWST_F300M_n']]).value*1e10)
            ews.append(get_EW_using_phot(row["JWST_F187N"], [row['JWST_F150W'], row['JWST_F300M']]).value*1e10)
        
        data = np.array([temp['Time'], ews, news, spews])
        np.save(f'{slug_output_dir}/{model_name}.npy', data)
    else:
        print('numpy file already exists, moving on...')

In [ ]:
dt = 1e5
N = 10
masses = [200, 2000, 5000, 10000, 20000, 50000]
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]
# Determine grid size (e.g. 2 rows x 3 columns for 6 plots)
n_cols = 3
n_rows = int(np.ceil(len(masses) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
fig2, ax2 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax2 = ax2.flatten()
fig3, ax3 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax3 = ax3.flatten()
fig4, ax4 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax4 = ax4.flatten()
fig5, ax5 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax5 = ax5.flatten()
fig6, ax6 = plt.subplots(n_rows, n_cols, figsize=(15, 8), sharex=True, sharey=False)
ax6 = ax6.flatten()
for i, m in enumerate(masses):
    model_name = f'10_trials_dt_1e5_m_{m}'
    data = np.load(f'{slug_output_dir}/{model_name}.npy')
    
    ax = axes[i]
    ax2_i = ax2[i]
    ax3_i = ax3[i]
    ax4_i = ax4[i]
    ax5_i = ax5[i]
    ax6_i = ax6[i]
    chunk_data = []


    for chunk in range(N):
        start = int(1e7/dt)*chunk
        ax.plot(data[0][start:start+int(1e7/dt)], data[1][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax.plot(data[0][start:start+int(1e7/dt)], data[2][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        ax2_i.plot(data[0][start:start+int(1e7/dt)], data[3][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax2_i.plot(data[0][start:start+int(1e7/dt)], data[4][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        #ax3_i.plot(data[0][start:start+int(1e7/dt)], data[5][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax3_i.plot(data[0][start:start+int(1e7/dt)], data[6][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        #ax4_i.plot(data[0][start:start+int(1e7/dt)], data[7][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax4_i.plot(data[0][start:start+int(1e7/dt)], data[8][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        #ax5_i.plot(data[0][start:start+int(1e7/dt)], data[9][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax5_i.plot(data[0][start:start+int(1e7/dt)], data[10][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[11][start:start+int(1e7/dt)], linewidth = 1, color = colors[chunk])
        #ax6_i.plot(data[0][start:start+int(1e7/dt)], data[12][start:start+int(1e7/dt)], linewidth = 0.5, color = colors[chunk])
        chunk_data.append(data[1][start:start+int(1e7/dt)])
        
    chunk_data = np.array(chunk_data)  # shape: (100, 10)
    
    # Calculate median across chunks for each timestep
    median_y = np.nanmedian(chunk_data, axis=0)
    
    # Overplot median track
    #ax.plot(data[0][start:start+int(1e7/dt)], median_y, color='red', linewidth=3, label='Median')
    
    ax.set_title(model_name)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('EW in Angstroms')
    ax2_i.set_title(model_name)
    ax2_i.set_xlabel('Age (years)')
    ax2_i.set_ylabel('QH0')
    ax3_i.set_title(model_name)
    ax3_i.set_xlabel('Age (years)')
    ax3_i.set_ylabel('f187')
    ax4_i.set_title(model_name)
    ax4_i.set_xlabel('Age (years)')
    ax4_i.set_ylabel('f150 level')
    ax5_i.set_title(model_name)
    ax5_i.set_xlabel('Age (years)')
    ax5_i.set_ylabel('f300')
    ax6_i.set_title(model_name)
    ax6_i.set_xlabel('Age (years)')
    ax6_i.set_ylabel('max_star_mass')
    ax.legend(fontsize=6)
# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
slug_input_file = '/project/galaxies/tjuchau/projects/EW_vs_Age/slug_tests/testing_cluster_mass_50000.slugin'
model_name = 'testing_cluster_mass_50000'
os.system(f"conda run -n Modeling {slug_path} {slug_input_file}")
temp = read_all_files(model_name, slug_output_dir='/project/galaxies/tjuchau/projects/EW_vs_Age/slug_tests')
rows = [dict(zip(temp.keys(), values)) for values in zip(*temp.values())]
ews = []

for row in rows:
    ews.append(get_EW_using_phot(row['JWST_F187N'], [row['JWST_F150W'], row['JWST_F300M']]).value*1e10)
data = np.array([temp['Time'], ews])

chunk_data = []
for chunk in range(100):
    start = 10*chunk
    plt.plot(data[0][start:start+10], data[1][start:start+10], linewidth = 0.1, color = 'black')
    chunk_data.append(data[1][start:start+10])
chunk_data = np.array(chunk_data)  # shape: (100, 10)

# Calculate median across chunks for each timestep
median_y = np.nanmedian(chunk_data, axis=0)

# Overplot median track
plt.plot(data[0][start:start+10], median_y, color='red', linewidth=3, label='Median')

plt.title(f'Mass = {50000}')
plt.xlabel('Age (years)')
plt.ylabel('EW in Angstroms')
plt.show()